# 0. Library

In [12]:
import pandas as pd
from pathlib import Path
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp
from features import general_func as gf
import constants_data as cd

# Reload
importlib.reload(du)
importlib.reload(dp)
importlib.reload(gf)
importlib.reload(cd)

<module 'constants_data' from '/home/smira/myproject/detection_AD_with_VR_data/src/notebooks/../constants_data.py'>

# 1. Paths and Constants

In [13]:
# Constants and paths
parent_folder = Path("../..") # go 2 folder up= "../.."
df_path = parent_folder / "data" / "produced_csv" / "1.extracted_features.csv"
 
df = pd.read_csv(df_path)

df.head()

,Paitent_id,Age,Gender,dominant_hand,Sessions_Completed_out_of_4,Help_Rating_out_of_5,MoCA_Score,Tutorial_reading_time_happened,Tutorial_total_reading_time,Tutorial_mean_reading_time,...,Memory_not_dominant_hand_y_range,Memory_not_dominant_hand_z_range,Memory_dominant_hand_trigger_press_count,Memory_not_dominant_hand_trigger_press_count,Memory_dominant_hand_trigger_pressure_mean,Memory_not_dominant_hand_trigger_pressure_mean,Memory_dominant_hand_grip_count,Memory_not_dominant_hand_grip_count,Memory_dominant_hand_grip_mean,Memory_not_dominant_hand_grip_mean
0,P001,NaN,Female,Right,4,4,28.0,1,45.03,15.01,...,0.80,0.50,35,0,0.01,0.0,1275,0,0.40,0.0
1,P002,59.0,Female,Right,4,2,NaN,1,171.99,57.33,...,0.61,1.15,63,0,0.02,0.0,222,0,0.09,0.0
2,P003,82.0,Male,Right,4,1,26.0,1,338.75,112.92,...,0.75,0.61,12,0,0.00,0.0,277,0,0.09,0.0
3,P004,75.0,Male,Left,4,2,NaN,1,114.78,38.26,...,0.59,0.92,47,1,0.03,0.0,157,0,0.10,0.0
4,P005,62.0,Male,Right,4,1,27.0,1,152.90,50.97,...,0.58,0.48,28,0,0.02,0.0,125,0,0.08,0.0


In [14]:
df.shape

(40, 377)

In [15]:
df_cleaning = df.iloc[:21].copy()
df_cleaning.shape

(21, 377)

# 2. Data analzying

# 2. Data preprocessing

In [16]:
df_cleaning = dp.check_Nan_values(original_df= df_cleaning, missing_values_threshold=0.6)

Missing values found, number of columns with Nan values : 4
All Nan values are handled successfuly.


In [17]:
df_cleaning = dp.check_columns_type(df_without_Nan= df_cleaning)

Columns before one hot encoding : 

          Total columns :376, 
          Numerical columns : 373,  
          Categorical columns : 3, 
          Unkown columns : 0
        

 Columns after one hot encoding : 

          Total columns :375, 
          Numerical columns : 375,  
          Categorical columns : 0, 
          Unkown columns : 0
        
All Categorical columns are encoded successfuly.


In [18]:
df_cleaning = dp.check_constant_column(df_without_categorical_column=df_cleaning)

Columns before removing constant columns : 375
There are 28 constant columns, removing them...

 Columns after removing constant columns : 347
All constant columns are removed successfuly.


In [19]:
df_cleaning = dp.check_correlation(df_cleaning)

Number of columns before removing 347
No Nan correlation.
Number of small correlation 174
Number of redundant features 66
Number of total useless columns 240, Removing them...
All useless correlations are removed successfully.
Number of columns after removing 107


In [20]:
df_cleaning.head()

,Help_Rating_out_of_5,MoCA_Score,Tutorial_total_reading_time,Tutorial_max_reading_time,Tutorial_total_duration_hover,Tutorial_mean_duration_hover,Tutorial_max_duration_hover,Tutorial_total_count_press,Tutorial_median_duration_press,Tutorial_total_count_gaze,...,Memory_Yaw_std,Memory_Pitch_std,Memory_Roll_std,Memory_Yaw_range,Memory_dominant_hand_mean_speed,Memory_not_dominant_hand_mean_speed,Memory_dominant_hand_z_range,Memory_not_dominant_hand_y_range,Gender_Male,dominant_hand_Right
0,4,28.0,45.03,21.00,20.64,1.03,2.29,8,0.15,2,...,26.23,161.72,177.36,99.05,0.16,0.03,0.82,0.80,0,1
1,2,27.0,171.99,91.56,53.05,1.02,13.42,9,0.41,15,...,47.08,165.33,172.85,180.39,0.06,0.04,1.02,0.61,0,1
2,1,26.0,338.75,155.14,145.79,2.11,20.01,9,0.25,21,...,23.27,161.90,162.61,109.38,0.06,0.01,0.98,0.75,1,1
3,2,27.0,114.78,68.19,46.90,1.47,13.24,8,0.25,12,...,25.75,171.61,101.97,234.99,0.06,0.03,0.50,0.59,1,0
4,1,27.0,152.90,88.47,54.33,1.81,11.62,8,0.28,21,...,30.07,170.64,169.17,248.93,0.12,0.02,1.05,0.58,1,1


## 2.2 Pipelines

In [ ]:
# R² = 1 - (error_model / error_baseline)
"""y = df_cleaning['MoCA_Score']
df_cleaning = df_cleaning.drop(columns=['MoCA_Score'])
x = df_cleaning

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

scores = cross_val_score(baseline, x, y, cv=5)
print(scores.mean())"""

-1.9336542861603854


In [23]:
df_cleaning.columns

Index(['Help_Rating_out_of_5', 'Tutorial_total_reading_time',
       'Tutorial_max_reading_time', 'Tutorial_total_duration_hover',
       'Tutorial_mean_duration_hover', 'Tutorial_max_duration_hover',
       'Tutorial_total_count_press', 'Tutorial_median_duration_press',
       'Tutorial_total_count_gaze', 'Tutorial_intensity_gaze',
       ...
       'Memory_Yaw_std', 'Memory_Pitch_std', 'Memory_Roll_std',
       'Memory_Yaw_range', 'Memory_dominant_hand_mean_speed',
       'Memory_not_dominant_hand_mean_speed', 'Memory_dominant_hand_z_range',
       'Memory_not_dominant_hand_y_range', 'Gender_Male',
       'dominant_hand_Right'],
      dtype='str', length=106)

In [22]:
# Pipeline 1 — Baseline (reference)
# scaling → model
model = Ridge()
scaler = StandardScaler()
y = df_cleaning['MoCA_Score']
df_cleaning = df_cleaning.drop(columns=['MoCA_Score'])
x = df_cleaning

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.4, random_state=42)
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model.fit(X_train_scaled)
print(model.coef_, model.intercept_)
y_predict = model.predict(X_test_scaled)



KeyError: 'MoCA_Score'

In [ ]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Pipeline 2 — PCA (compression approach)
# scaling → PCA → model

In [ ]:
# Pipeline 3 — Feature selection (SelectKBest)
# scaling → SelectKBest → model


In [ ]:
# Pipeline 4 — Regularization (L1 / Lasso)
# scaling → L1 model

## 2.3 comparing pipelines
Pipeline → Cross-validation → compare scores